In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd
from mixing_utils import mix_signals

SINR_GRID = np.arange(-10, 22, 2)  # [-10, -8, ..., 20]
TARGET_LEN = 40960

In [ ]:
def generate_evaluation_grid(split_name):
    print(f"\n=== Generating Grid for: {split_name.upper()} ===")
    
    # Load Raw Split Data
    clean_path = get_unet_path(STAGE_SPLIT, split=split_name, signal=CommSignal2)
    clean_signals_raw = np.load(clean_path)
    
    # Load Interference Dictionary
    interferences = {
        'Barrage':   {'data': np.load(get_unet_path(STAGE_SPLIT, split=split_name, signal=BarrageSignal)), 'source': 'AWGN_Generator'},
        'Spot':      {'data': np.load(get_unet_path(STAGE_SPLIT, split=split_name, signal=EMISignal1)),    'source': EMISignal1},
        'Digital3':  {'data': np.load(get_unet_path(STAGE_SPLIT, split=split_name, signal=CommSignal3)),   'source': CommSignal3},
        'Digital5G': {'data': np.load(get_unet_path(STAGE_SPLIT, split=split_name, signal=CommSignal5G1)), 'source': CommSignal5G1}
    }
    
    int_types = list(interferences.keys())
    n_clean_files = len(clean_signals_raw)
    
    print(f"Loaded {n_clean_files} clean files.")
    print(f"Grid: {len(SINR_GRID)} SINR levels x {len(int_types)} Interference Types")
    print(f"Expected Output: {n_clean_files * len(SINR_GRID) * len(int_types)} samples.")

    mixed_signals = []
    metadata_rows = []
    
    global_counter = 0

    # The Nested Grid Loop
    # Loop 1: SINR (Difficulty)
    for sinr_db in tqdm(SINR_GRID, desc="SINR Grid"):
        
        # Loop 2: Interference Type (Category)
        for itype in int_types:
            int_data = interferences[itype]['data']
            int_source_name = interferences[itype]['source']
            
            # Loop 3: Clean Files (Content)
            for clean_idx in range(n_clean_files):
                
                # Prepare Clean Signal (Deterministic Crop)
                # We always take the first 40960 samples to ensure reproducibility
                raw_sig = clean_signals_raw[clean_idx]

                clean_len = raw_sig.shape[1]
                noise_len = int_data[0].shape[1]

                if raw_sig.shape[1] >= TARGET_LEN:
                    sig_clean = raw_sig[:, :TARGET_LEN]

                elif noise_len == clean_len:
                    sig_noise = sig_noise
                else:
                    # Failure (Noise is shorter than signal)
                    # We explicitly FORBID looping to prevent artifacts.
                    raise ValueError(
                        f"CRITICAL DATA ERROR: Interference source '{itype}') "
                        f"has length {noise_len}, which is shorter than the target {clean_len}. "
                        "Looping is forbidden."
                    )
                
                # Prepare Noise Signal (Deterministic Selection)
                # We select noise sequentially using the global counter
                # This ensures we rotate through the available noise files evenly
                noise_file_idx = global_counter % len(int_data)
                raw_noise = int_data[noise_file_idx]
                
                # Deterministic Crop for Noise (Start at 0)
                if raw_noise.shape[1] >= TARGET_LEN:
                    sig_noise = raw_noise[:, :TARGET_LEN]
                else:
                    raise ValueError(f"Noise file {itype} #{noise_file_idx} is too short for evaluation!")

                # Mix (Physics)
                sig_mixed, alpha = mix_signals(sig_clean, sig_noise, sinr_db)
                
                # Store
                mixed_signals.append(sig_mixed)
                metadata_rows.append({
                    'global_index': global_counter,
                    'sinr_db': round(sinr_db, 4),
                    'clean_frame_id': clean_idx,
                    'noise_frame_id': noise_file_idx,
                    'interference_type': itype,
                    'source': int_source_name,
                    'alpha': round(alpha, 6)
                })
                
                global_counter += 1

    # Save Artifacts
    output_dir = get_unet_path(STAGE_MIXED, split=split_name)
    output_npy = output_dir / f"{MIXED_DATASET}.npy"
    output_csv = output_dir / f"{MIXED_METADATA}.csv"
    
    # Save mixed tensor
    np.save(output_npy, np.array(mixed_signals))
    
    # Save metadata
    df = pd.DataFrame(metadata_rows)
    df.to_csv(output_csv, index=False)
    
    print(f"Saved {len(mixed_signals)} samples to {output_npy}")
    print(f"Saved Metadata to {output_csv}")


In [ ]:
generate_evaluation_grid(VAL)
generate_evaluation_grid(TEST)